# Trabalho Prático — Análise Exploratória e Pré-processamento de Dados
### Disciplina: Aprendizado de Máquina — IFMA Campus Coelho Neto
**Professor:** Bruno Vicente
**Dupla:** Rubens Dutra de Mesquita Filho e Pedro Alexandre
**Base de dados:** Bank Marketing (UCI Machine Learning Repository)
**Data:** Setembro/2026

---


## 0. Configuração inicial
Importação das bibliotecas utilizadas ao longo do notebook.

In [ ]:
# Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import io

# Bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações gerais de exibição
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

print("Bibliotecas carregadas com sucesso.")

## 1. Definição do problema

**Contexto e domínio:** a base reúne dados de campanhas de telemarketing de uma instituição
bancária portuguesa. As campanhas consistiram em ligações telefônicas oferecendo a clientes a
adesão a um depósito a prazo (aplicação financeira). Cada instância representa um cliente
contatado, descrito por atributos demográficos, financeiros e características do próprio contato
telefônico.

**Atributo-alvo:** `y`, atributo categórico binário (`yes` = aderiu ao depósito, `no` = não
aderiu).

**Tipo de tarefa:** classificação binária.

**Relevância:** um modelo capaz de prever a adesão ao produto ajuda a instituição financeira a
priorizar quais clientes contatar, reduzindo o custo operacional de campanhas de telemarketing e
aumentando a taxa de conversão. Beneficia tanto o banco (eficiência de campanha) quanto o cliente
(menos contatos desnecessários).

**Observação metodológica importante:** o atributo `duration` (duração da ligação, em segundos)
só é conhecido **depois** que a ligação acontece — ou seja, não estaria disponível no momento em
que um modelo precisaria decidir se vale a pena ligar para o cliente. Isso é conhecido na
literatura como um caso de *data leakage* em potencial. Como este trabalho não treina modelo,
mantemos `duration` na análise exploratória (ele é informativo sobre o processo de contato), mas
registramos essa ressalva para deixar claro que, numa etapa futura de modelagem, esse atributo
exigiria tratamento especial.

## 2. Coleta e compreensão dos dados

**Origem:** UCI Machine Learning Repository — *Bank Marketing* (doada por S. Moro, P. Cortez e
P. Rita). Link: https://archive.ics.uci.edu/dataset/222/bank+marketing

**Licença:** Creative Commons Attribution 4.0 International (CC BY 4.0).

**Citação (ABNT):**
> MORO, S.; CORTEZ, P.; RITA, P. **Bank Marketing**. UCI Machine Learning Repository, 2012. DOI: https://doi.org/10.24432/C5K306. Disponível em: https://archive.ics.uci.edu/dataset/222/bank+marketing. Acesso em: 23 set. 2026.

**Artigo de referência:** MORO, S.; CORTEZ, P.; RITA, P. A data-driven approach to predict the
success of bank telemarketing. **Decision Support Systems**, v. 62, p. 22-31, 2014.

Usamos aqui a versão `bank-full.csv` (45.211 instâncias, 16 atributos + alvo), a versão completa
com o conjunto "clássico" de 17 colunas.

In [ ]:
# Carregamento dos dados diretamente da fonte oficial (garante reprodutibilidade)
# O UCI distribui esta base como um .zip contendo mais de um arquivo,
# por isso extraímos o bank-full.csv em memória antes de carregar no pandas.
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"

resposta = urllib.request.urlopen(url)
arquivo_zip = zipfile.ZipFile(io.BytesIO(resposta.read()))

with arquivo_zip.open("bank-full.csv") as f:
    df = pd.read_csv(f, sep=";")

print(f"Dimensões da base: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
# Tipos de dados de cada coluna, conforme o pandas os interpretou
df.dtypes

**Interpretação:** o pandas reconheceu automaticamente 7 colunas numéricas (`age`, `balance`,
`day`, `duration`, `campaign`, `pdays`, `previous`) e 10 colunas categóricas/texto (`job`,
`marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, `poutcome` e o
alvo `y`). Isso confirma a mistura de tipos exigida pelo trabalho — e, diferente da base anterior,
aqui **todos os nomes de coluna são documentados oficialmente pela fonte**, sem necessidade de
hipóteses de terceiros.

### Dicionário de dados

| Coluna | Tipo | Descrição | Valores possíveis / Unidade |
|---|---|---|---|
| `age` | Numérico (discreto) | Idade do cliente | anos |
| `job` | Categórico | Tipo de emprego | 12 categorias (admin, blue-collar, management, ... , unknown) |
| `marital` | Categórico | Estado civil | married, divorced, single (*divorced* inclui viúvo(a)) |
| `education` | Categórico | Nível de escolaridade | unknown, primary, secondary, tertiary |
| `default` | Categórico binário | Se o cliente possui crédito em situação de inadimplência | yes, no |
| `balance` | Numérico (contínuo) | Saldo médio anual da conta | euros |
| `housing` | Categórico binário | Se o cliente possui empréstimo habitacional | yes, no |
| `loan` | Categórico binário | Se o cliente possui empréstimo pessoal | yes, no |
| `contact` | Categórico | Tipo de contato usado na ligação | unknown, telephone, cellular |
| `day` | Numérico (discreto) | Dia do mês do último contato | 1-31 |
| `month` | Categórico | Mês do último contato | jan a dec |
| `duration` | Numérico (contínuo) | Duração do último contato | segundos (ver ressalva de *data leakage* na Etapa 1) |
| `campaign` | Numérico (discreto) | Número de contatos feitos nesta campanha para este cliente | contagem (inclui o último contato) |
| `pdays` | Numérico (discreto) | Dias desde o último contato de uma campanha anterior | dias (`-1` = cliente nunca contatado antes) |
| `previous` | Numérico (discreto) | Número de contatos feitos antes desta campanha | contagem |
| `poutcome` | Categórico | Resultado da campanha de marketing anterior | unknown, other, failure, success |
| `y` | **Categórico binário (alvo)** | Cliente aderiu ao depósito a prazo? | yes, no |

*Fonte: elaborado pelos autores, com base na documentação oficial do UCI (archive.ics.uci.edu/dataset/222/bank+marketing).*

> **Observação sobre a categoria "unknown":** a fonte oficial declara que a base **não possui
> valores ausentes (NaN)**. No entanto, quatro colunas categóricas (`job`, `education`, `contact`,
> `poutcome`) usam o valor `"unknown"` como uma categoria válida — que, na prática, funciona como
> um dado faltante disfarçado. Trataremos isso formalmente na Etapa 3 (diagnóstico de qualidade) e
> na Etapa 4 (pré-processamento), já que o pandas não vai contar essas ocorrências como `NaN`
> automaticamente.

In [ ]:
# Verificação de valores ausentes "tradicionais" (NaN)
print("Valores ausentes (NaN) por coluna:")
print(df.isnull().sum().sum(), "no total (esperado: 0, conforme documentação oficial)")

In [ ]:
# Verificação da categoria "unknown", que funciona como ausência disfarçada
colunas_com_unknown = ["job", "education", "contact", "poutcome"]

resumo_unknown = pd.DataFrame({
    "Ocorrências de 'unknown'": [(df[col] == "unknown").sum() for col in colunas_com_unknown],
    "Percentual (%)": [round((df[col] == "unknown").mean() * 100, 2) for col in colunas_com_unknown]
}, index=colunas_com_unknown)

resumo_unknown.sort_values("Ocorrências de 'unknown'", ascending=False)

**Interpretação:** a coluna `poutcome` tem a maior concentração de "unknown" (a maioria dos
clientes nunca havia sido contatada em campanha anterior, o que é coerente com `pdays = -1`).
`contact` também tem uma fração relevante de valores desconhecidos. Isso confirma que a base
atende ao critério de "dados reais com problemas de qualidade" exigido pelo trabalho, mesmo sem
ausentes no sentido técnico de `NaN`.

In [ ]:
# Verificação de linhas duplicadas
print(f"Linhas duplicadas: {df.duplicated().sum()}")

In [ ]:
# Distribuição inicial do atributo-alvo (visão rápida — a análise completa fica na Etapa 3)
df["y"].value_counts()


In [ ]:
# Percentual de cada classe do alvo — já dá pra ver o desbalanceamento
df["y"].value_counts(normalize=True).round(4) * 100